<a href="https://colab.research.google.com/github/CH3MLON/Data_GenAI_training/blob/main/day9/Day09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install groq -q

In [5]:
import sqlite3
import pandas as pd
import re
from groq import Groq
import os

In [6]:
os.environ["GROQ_API_KEY"]="your_api_key"
client=Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL="llama-3.1-8b-instant"
print("client model initialized")

client model initialized


In [7]:
import io
ds=pd.read_csv("/content/student_performance.csv")
ds.shape

(30, 13)

In [8]:
conn = sqlite3.connect("college.db")
ds.to_sql("students",conn, if_exists="replace",index=False)

30

In [9]:
test_ds=pd.read_sql_query("SELECT COUNT(*) as total_rows from students",conn)
print(f"Verification: {test_ds['total_rows'][0]}")

Verification: 30


In [10]:
def get_schema(conn, table_name="students"):
  "this is for your description"
  cursor=conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")
  columns=cursor.fetchall()
  schema_lines=[f"Table: {table_name}"]
  schema_lines.append("Columns:")

  for col in columns:
    schema_lines.append(f"({col[1]})({col[2]})")
  cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
  sample_rows=cursor.fetchall()
  schema_lines.append("\n Sample rows(first 3): ")

  for row in sample_rows:
    schema_lines.append(f" {row}")
  return "\n".join(schema_lines)

schema= get_schema(conn)
print(schema)

Table: students
Columns:
(student_id)(INTEGER)
(name)(TEXT)
(age)(INTEGER)
(gender)(TEXT)
(department)(TEXT)
(semester)(INTEGER)
(math_score)(INTEGER)
(science_score)(INTEGER)
(english_score)(INTEGER)
(programming_score)(INTEGER)
(attendance_percentage)(INTEGER)
(city)(TEXT)
(admission_year)(INTEGER)

 Sample rows(first 3): 
 (1001, 'Aarav Sharma', 19, 'Male', 'Computer Science', 2, 85, 78, 72, 91, 92, 'Mumbai', 2023)
 (1002, 'Priya Patel', 20, 'Female', 'Computer Science', 2, 76, 82, 88, 79, 87, 'Ahmedabad', 2023)
 (1003, 'Rohit Verma', 19, 'Male', 'Electronics', 2, 65, 74, 61, 55, 78, 'Delhi', 2023)


In [15]:
def generate_sql(user_question, schema_text, client, model):
  system_prompt=f"""You are an expert SQL assistant.
  You are connected to a SQLite database with the following structure

  {schema_text}

  RULES:
  1.generate only a valid salite sql query.
  2.do not include any explanation or text - only the sql query
  3.do not use markdown code blocks. return the rqw sql only
  4.the table name is students
  5.only use columns that exist in  the schema
  6.use single quotes for string values in WHERE clause (example: WHERE subject='Programming')
  7.if the user asks for top N, use ORDER BY marks DESC LIMIT N.
  """

  response=client.chat.completions.create(
      model=model, #from where you are gonna get your response from
      messages=[
          {"role":"system", "content":system_prompt},
          {"role":"user", "content":user_question}
      ],

      temperature=0.0
  )
  sql_query=response.choices[0].message.content
  return sql_query

question="I wan the number of students who achieved score more than the average from programming,science,math"
print(f"question: {question}")
sql=generate_sql(question,schema,client,MODEL)
print(f"\nsql query: {sql}")

question: I wan the number of students who achieved score more than the average from programming,science,math

sql query: SELECT COUNT(*) FROM students WHERE programming_score + science_score + math_score > ( SELECT AVG(programming_score + science_score + math_score) FROM students )
